title: "Markets"author: "Christian Kruse"date: "`r Sys.Date()`"output: html_document

In [ ]:
knitr::opts_chunk$set(echo = FALSE,message = FALSE,warning = FALSE)options(scipen=999)

In [ ]:
if (!require(pacman)) { install.packages("pacman") }pacman::p_load(fredr,               dplyr,               tidyr,               lubridate,               caret,               partykit,               pROC,               gganimate,               devtools,               gridExtra,               httr,               DT,               httr,               glue,               visNetwork,               dkstat,               data.table,               pROC,               rPref,               progress,               quantmod,               rvest,               ggrepel)source(file = "money_theme.R")# library("devtools")# install_github("rOpenGov/dkstat")library(dkstat)

In [ ]:
readRenviron(path = "Renviron.site")fredr::fredr_set_key(key = Sys.getenv("FRED_API"))

# MarketsPrice index DK

In [ ]:
# Indexed to 2015df_PRIS114 = dkstat::dst_get_data(table = "PRIS114",                     VAREGR = "00 Nettoprisindeks i alt",                     ENHED = "Indeks",                     Tid = "*") %>%   arrange(TID) %>%   dplyr::mutate(Index=value/100) %>%   tidyr::complete(TID=seq.Date(from=min(TID,na.rm=T),to=Sys.Date(),by="1 day")) %>%   tidyr::fill(TID,.direction = "down")  %>%   tidyr::fill(value,.direction = "down")   %>%   tidyr::fill(VAREGR,.direction = "down")   %>%   tidyr::fill(ENHED,.direction = "down")    %>%   tidyr::fill(Index,.direction = "down") %>%   dplyr::select(TID,value) %>%   dplyr::rename(date=TID) %>%   dplyr::mutate(series_id="PRICE_INDEX",                currency="DKK")

Price index US

In [ ]:
df_CPIAUCNS = fredr::fredr("CPIAUCNS") %>%     dplyr::select(series_id,                  date,                  value) %>%     tidyr::complete(date=seq.Date(from=min(date,na.rm=T),to=Sys.Date(),by="1 day")) %>%     tidyr::fill(value,.direction = "down") %>%     tidyr::fill(series_id,.direction = "down") %>%   dplyr::mutate(index=ifelse(date==ymd("2015-01-01"),value,NA)) %>%   tidyr::fill(index,.direction = "downup") %>%   dplyr::mutate(value=100*value/index,                series_id="PRICE_INDEX",                currency="USD") %>%   dplyr::select(-index)

Index 1 (To keep nominal prices)

In [ ]:
df_nominal = expand.grid(currency=c("USD","DKK"),                         value=1,                         series_id="NOMINAL",                         date=seq.Date(min(df_CPIAUCNS$date),max(df_CPIAUCNS$date),by="1 day"))

Gold price, Silver Price, Oil Price, Bitcoin, USD

In [ ]:
series_ids_yahoo = unique(c("CL=F",                            "GC=F",                            "SI=F",                            "BTC-USD",                            "DKKUSD=X",                            "USDDKK=X"))df_macro_yahoo = rbindlist(lapply(series_ids_yahoo,function(ticker) {      temp_df = getSymbols(ticker, env = NULL) %>%       as.data.frame(.) %>%       dplyr::mutate(Date=row.names(.)) %>%       dplyr::mutate(Date=gsub("X","",Date)) %>%       dplyr::mutate(Date=ymd(Date)) %>%       dplyr::select(c(7,1)) %>%       setNames(.,c("date","value")) %>%       dplyr::mutate(series_id=ticker) %>%       tidyr::complete(date=seq.Date(from=min(date,na.rm=T),to=Sys.Date(),by="1 day")) %>%      tidyr::fill(value,.direction = "down")  %>%      tidyr::fill(series_id,.direction = "down")            row.names(temp_df) = NULL      return( temp_df )}))# USDdf_macro_yahoo_usd = df_macro_yahoo %>%   filter(!series_id=="USDDKK=X") %>%   dplyr::mutate(currency="USD")# DKKdf_macro_yahoo_dkk = df_macro_yahoo %>%   inner_join( df_macro_yahoo %>% filter(series_id=="USDDKK=X") %>% dplyr::select(date,value) %>% dplyr::rename(usd=value)) %>%   filter(!series_id=="DKKUSD=X") %>%   dplyr::mutate(value=value*usd,                currency="DKK") %>%   dplyr::select(-usd)df_macros = df_macro_yahoo_usd %>%   bind_rows( df_macro_yahoo_dkk ) %>%   bind_rows(df_CPIAUCNS) %>%   bind_rows(df_PRIS114) %>%   bind_rows(df_nominal)

TickersShort:- SHV: iShares Short Treasury Bond ETF- TBF: ProShares Short 20+ Yr Treasury- TTT: UltraPro Short 20+ Year Treasury- TBX: Short 7-10 Year Treasury- PST: UltraShort 7-10 Year Treasury- TBT: UltraShort 20+ Year TreasuryLong:- MBB: iShares MBS ETF- GOVT: iShares U.S. Treasury Bond ETF- SGOV: iShares 0-3 Month Treasury Bond ETF- SHY: iShares 1-3 Year Treasury Bond ETF- IEI: iShares 3-7 Year Treasury Bond ETF- IEF: iShares 7-10 Year Treasury Bond ETF- UST: Ultra 7-10 Year Treasury- TLH: iShares 10-20 Year Treasury Bond ETF- TLT: iShares 20+ Year Treasury Bond ETF- UBT: Ultra 20+ Year TreasuryCommodities:- IAU: iShares Gold Trust- SLV: iShares Silver Trust

In [ ]:
series_ids_yahoo = unique(c("GC=F",                            "SI=F",                            "BTC-USD",                            "ETH-USD",                            # "^RUT",                            "^W5000",                            "^IXIC",                            "^GSPC",                                                        "SHV",                            "TBF",                            "TTT",                            "TBX",                            "PST",                            "TBT",                                                        "XLK",                            "XLV",                            "XLY",                            "XLP",                            "XLU",                            "XLF",                            "XLI",                            "XLB",                            "XLE",                            "XLRE",                            "XLC",                                                        "REW",                            "RXD",                            "SCC",                            "SDP",                            "SKF",                            "SIJ",                            "SMN",                            "DUG",                            "SRS",                                                        "MBB",                            "GOVT",                            "SGOV",                            "SHY",                            "IEI",                            "IEF",                            "UST",                            "UBT",                            "TLT",                                                        "^OMXC25"))df_tickers_yahoo = rbindlist(lapply(series_ids_yahoo,function(ticker) {      temp_df = getSymbols(ticker, env = NULL) %>%       as.data.frame(.) %>%       dplyr::mutate(Date=row.names(.)) %>%       dplyr::mutate(Date=gsub("X","",Date)) %>%       dplyr::mutate(Date=ymd(Date)) %>%       dplyr::select(c(7,1)) %>%       setNames(.,c("date","price")) %>%       dplyr::mutate(ticker=ticker) %>%       tidyr::complete(date=seq.Date(from=min(date,na.rm=T),to=Sys.Date(),by="1 day")) %>%      tidyr::fill(price,.direction = "down")  %>%      tidyr::fill(ticker,.direction = "down")            row.names(temp_df) = NULL      return( temp_df )})) 

In [ ]:
df_adjusted = df_tickers_yahoo %>%   inner_join( df_macros ) %>%   dplyr::mutate(index=price/value) %>%   dplyr::select(-price,-value)

Plot function

In [ ]:
draw_plots = function(data,lines=NA) {    ticker_ = unique(data$ticker)  temp_df = data %>% filter(ticker==ticker_)    # Lines  df_lines = temp_df %>%     filter(series_id=="NOMINAL")     if (unique(!is.na(lines))) {  df_lines = rbindlist(lapply(seq(length(lines)),function(line) {        df_lines %>%       dplyr::mutate(series_id="SUPPORT",                    predicted=predict(        df_lines %>%           filter(date %in% lines[[line]]) %>%           lm(index~date,data=.),        .)) %>%       dplyr::mutate(category=paste0("line",line))      }))  } else {  df_lines = data.frame()  }  # All time plotp1 = temp_df %>%   filter(series_id=="NOMINAL") %>%   ggplot(.,aes(x=date,y=index)) +  geom_line() +  theme_money_printer_go_brrr(12) +  theme(legend.position = "bottom") +  scale_x_date(date_labels = "%Y",date_breaks = "5 years") +  scale_y_continuous(limits = c(0,NA)) +  labs(x=NULL,y=NULL,title=glue("{ticker_}"),subtitle="All Time")if (nrow(df_lines)>0) {  p1 = p1 +     geom_line(data=df_lines,aes(x=date,y=predicted,color=category,group=category),linetype=2)}# Last yearp2 = temp_df %>%   filter(series_id=="NOMINAL") %>%    filter(date>=Sys.Date()-years(1)) %>%   ggplot(.,aes(x=date,y=index)) +  geom_line() +  theme_money_printer_go_brrr(12) +  theme(legend.position = "bottom") +  scale_x_date(date_labels = "%b",date_breaks = "1 month",limits = c(Sys.Date()-years(1),NA)) +  geom_vline(xintercept = Sys.Date(),linetype=2) +  labs(x=NULL,y=NULL,subtitle="Last Year")# Indicator heatmapp2b = temp_df %>%   filter(series_id=="NOMINAL") %>%   arrange((date)) %>%   dplyr::mutate(value_rsi=TTR::RSI(index),                value_macd=TTR::MACD(index)[,1]-TTR::MACD(index)[,2],                value_cci=TTR::CCI(index),                value_goldencross=TTR::SMA(index,n=50)-TTR::SMA(index,n=200)) %>%   dplyr::mutate(value_lastdate=ifelse(index<last(index) & !date==max(date),as.Date(date),as.Date(NA))) %>%   dplyr::mutate(value_lastdate=max(value_lastdate,na.rm=T)) %>%   dplyr::mutate(value_lastdate=as.numeric(date-as.Date(value_lastdate))) %>%   tidyr::fill(value_lastdate,.direction = "down") %>%   dplyr::mutate(p_rsi=pnorm(value_rsi,mean=mean(value_rsi,na.rm=T),sd=sd(value_rsi,na.rm=T)),                p_macd=pnorm(value_macd,mean=mean(value_macd,na.rm=T),sd=sd(value_macd,na.rm=T)),                p_cci=pnorm(value_cci,mean=mean(value_cci,na.rm=T),sd=sd(value_cci,na.rm=T)),                p_goldencross=pnorm(value_goldencross,mean=mean(value_goldencross,na.rm=T),sd=sd(value_goldencross,na.rm=T)),                p_pnorm=pnorm(index,mean=mean(tail(index,3*365),na.rm=T),sd=sd(tail(index,3*365),na.rm=T)),                 p_lastdate=0.5) %>%   filter(date==max(date)) %>%   dplyr::select(-currency,-index) %>%   gather(stat,val,value_rsi:p_lastdate) %>%   separate(stat,into=c("type","metric")) %>%   spread(type,val) %>%   ggplot(aes(x=metric,y=ticker,fill=p)) +  geom_tile() +  geom_text(aes(label=paste0(scales::number(value,0.01)," (",scales::percent(p,0.1),")")),color="white") +   scale_fill_gradientn(colours=hcl.colors(10, palette = "ag_GrnYl"),                       na.value = "transparent",                       breaks=c(0,0.5,1),                       labels=c("Minimum",0.5,"Maximum"),                       limits=c(0,1)) +  theme_money_printer_go_brrr(12) +  theme(legend.position="null",        axis.text.y=element_blank()) +  labs(x=NULL,y=NULL)  # SMAp3 = temp_df %>%   filter(series_id=="NOMINAL") %>%   arrange(date) %>%   dplyr::mutate(sma200=TTR::SMA(index,n=200)) %>%   dplyr::mutate(index=index-sma200) %>%   filter(date>=Sys.Date()-years(1)) %>%   ggplot(.,aes(x=date,y=index)) +  geom_line() +  theme_money_printer_go_brrr(12) +  scale_x_date(date_labels = "%b",date_breaks = "1 month") +  geom_hline(yintercept = 0,linetype=2) +  labs(x=NULL,y=NULL,subtitle="SMA 200")# RSIp4 = temp_df %>%   filter(series_id=="NOMINAL") %>%   dplyr::mutate(rsi=TTR::RSI(index,n=30)) %>%   filter(date>=Sys.Date()-years(1)) %>%   ggplot(.,aes(x=date,y=rsi)) +  geom_line() +  theme_money_printer_go_brrr(12) +  scale_x_date(date_labels = "%b",date_breaks = "1 month") +  geom_hline(yintercept = c(20,70),linetype=2) +  labs(x=NULL,y=NULL,subtitle="RSI")# Real terms (US)p5 = temp_df %>%   filter(series_id=="PRICE_INDEX") %>%   ggplot(.,aes(x=date,y=index)) +  geom_line() +  theme_money_printer_go_brrr(12) +  scale_x_date(date_labels = "%Y",date_breaks = "1 year") +  labs(x=NULL,y=NULL,subtitle="Real Terms (US)")# US Dollarsp6 = temp_df %>%   filter(series_id=="USDDKK=X") %>%   ggplot(.,aes(x=date,y=index)) +  geom_line() +  theme_money_printer_go_brrr(12) +  scale_x_date(date_labels = "%Y",date_breaks = "1 year") +  labs(x=NULL,y=NULL,subtitle="DKK")# Oilp7 = temp_df %>%   filter(series_id=="CL=F") %>%   ggplot(.,aes(x=date,y=index)) +  geom_line() +  theme_money_printer_go_brrr(12) +  scale_x_date(date_labels = "%Y",date_breaks = "1 year") +  scale_y_continuous(limits=c(0,150)) +  labs(x=NULL,y=NULL,subtitle="Crude Oil")# Silverp8 = temp_df %>%   filter(series_id=="SI=F") %>%   ggplot(.,aes(x=date,y=index)) +  geom_line() +  theme_money_printer_go_brrr(12) +  scale_x_date(date_labels = "%Y",date_breaks = "1 year") +  labs(x=NULL,y=NULL,subtitle="Silver")# Goldp9 = temp_df %>%   filter(series_id=="GC=F") %>%   ggplot(.,aes(x=date,y=index)) +  geom_line() +  theme_money_printer_go_brrr(12) +  scale_x_date(date_labels = "%Y",date_breaks = "1 year") +  labs(x=NULL,y=NULL,subtitle="Gold")  # Index  p10 = temp_df  %>%     filter(series_id %in% c("PRICE_INDEX","NOMINAL")) %>%     group_by(series_id) %>%     dplyr::mutate(nominal_2015=ifelse(date==ymd("2015-01-01"),index,NA)) %>%     tidyr::fill(nominal_2015,.direction = "downup") %>%     ungroup() %>%     dplyr::mutate(index=100*index/nominal_2015) %>%     dplyr::select(-nominal_2015) %>%     ggplot(aes(x=date,y=index,color=series_id,group=series_id)) +    geom_line() +    theme_money_printer_go_brrr(12) +    scale_x_date(date_labels = "%Y",date_breaks = "1 year") +    labs(x=NULL,y=NULL,subtitle="Indexed 2015")  # Above/below price index  p11 = temp_df  %>%     filter(series_id %in% c("PRICE_INDEX","NOMINAL")) %>%     group_by(series_id) %>%     dplyr::mutate(nominal_2015=ifelse(date==ymd("2015-01-01"),index,NA)) %>%     tidyr::fill(nominal_2015,.direction = "downup") %>%     ungroup() %>%     dplyr::mutate(index=100*index/nominal_2015) %>%     dplyr::select(-nominal_2015) %>%     dplyr::mutate(priceindex=ifelse(series_id=="PRICE_INDEX",index,NA)) %>%     tidyr::fill(priceindex,.direction = "downup") %>%     dplyr::mutate(index=index-priceindex) %>%     filter(!series_id=="PRICE_INDEX") %>%     ggplot(aes(x=date,y=index,color=series_id,group=series_id)) +    geom_line() +    theme_money_printer_go_brrr(12) +    scale_x_date(date_labels = "%Y",date_breaks = "1 year") +    labs(x=NULL,y=NULL,subtitle="Above/below price trend") +    geom_hline(yintercept = 0,linetype=2)p_all = grid.arrange(p1,p2,p2b,p3,p4,                     # p5,                     p6,p7,p8,p9,p10,p11,ncol=1)print(p_all)}

## SP500

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="^GSPC",currency=="USD"),           lines = list(c(ymd("2020-03-23"),ymd("2023-10-30")),                        c(ymd("2018-01-20"),ymd("2022-01-01")))           )

## NASDAQ

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="^IXIC",currency=="USD"),           lines = list(c(ymd("2020-03-23"),ymd("2023-01-03"))))

## OMX C25

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="^OMXC25",currency=="DKK"),           lines = list(c(ymd("2020-03-23"),ymd("2023-10-30")),                        c(ymd("2018-01-20"),ymd("2022-01-01"))))

## RUSSELL 2000

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="^RUT",currency=="USD"))

## WILLSHIRE

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="^W5000",currency=="USD"))

## BITCOIN

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="BTC-USD",currency=="USD"))

## GOLD

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="GC=F",currency=="USD"))

## Silver

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="SI=F",currency=="USD"))

# Bond ETFsShort:- SHV: iShares Short Treasury Bond ETF

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="SHV",currency=="USD"))

- TBF: ProShares Short 20+ Yr Treasury

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="TBF",currency=="USD"))

- TTT: UltraPro Short 20+ Year Treasury

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="TTT",currency=="USD"))

- TBX: Short 7-10 Year Treasury

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="TBX",currency=="USD"))

- PST: UltraShort 7-10 Year Treasury

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="PST",currency=="USD"))

- TBT: UltraShort 20+ Year Treasury

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="TBT",currency=="USD"))

Long:- MBB: iShares MBS ETF

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="MBB",currency=="USD"))

- GOVT: iShares U.S. Treasury Bond ETF

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="GOVT",currency=="USD"))

- SGOV: iShares 0-3 Month Treasury Bond ETF

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="SGOV",currency=="USD"))

- SHY: iShares 1-3 Year Treasury Bond ETF

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="SHY",currency=="USD"))

- IEI: iShares 3-7 Year Treasury Bond ETF

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="IEI",currency=="USD"))

- IEF: iShares 7-10 Year Treasury Bond ETF

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="IEF",currency=="USD"))

- UST: Ultra 7-10 Year Treasury

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="UST",currency=="USD"))

- TLH: iShares 10-20 Year Treasury Bond ETF

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="TBT",currency=="USD"))

- TLT: iShares 20+ Year Treasury Bond ETF

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="TLT",currency=="USD"))

- UBT: Ultra 20+ Year Treasury

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="UBT",currency=="USD"))

Commodities:- IAU: iShares Gold Trust- SLV: iShares Silver Trust

# Sector ETFsLong:- Technology: Technology Select Sector SPDR Fund (XLK)

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="XLK",currency=="USD"))

- Health Care: Health Care Select Sector SPDR Fund (XLV)

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="XLV",currency=="USD"))

- Consumer Discretionary: Consumer Discretionary Select Sector SPDR Fund (XLY)

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="XLY",currency=="USD"))

- Consumer Staples: Consumer Staples Select Sector SPDR Fund (XLP)

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="XLP",currency=="USD"))

- Utilities: Utilities Select Sector SPDR Fund (XLU)

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="XLU",currency=="USD"))

- Financials: Financial Select Sector SPDR Fund (XLF)

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="XLF",currency=="USD"))

- Industrials: Industrial Select Sector SPDR Fund (XLI)

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="XLI",currency=="USD"))

- Materials: Materials Select Sector SPDR Fund (XLB)

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="XLB",currency=="USD"))

- Energy: Energy Select Sector SPDR Fund (XLE)

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="XLE",currency=="USD"))

- Real Estate: Real Estate Select Sector SPDR Fund (XLRE)

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="XLRE",currency=="USD"))

- Communication Services: Communication Services Select Sector SPDR Fund (XLC)

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="XLC",currency=="USD"))

Shorts:Technology Sector: ProShares UltraShort Technology - REW (Note: This is 2x leveraged)

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="REW",currency=="USD"))

Health Care Sector: ProShares UltraShort Health Care - RXD (Note: This is 2x leveraged)

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="RXD",currency=="USD"))

Consumer Discretionary Sector: ProShares UltraShort Consumer Services - SCC (Note: This is 2x leveraged)

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="SCC",currency=="USD"))

Utilities Sector: ProShares UltraShort Utilities - SDP (Note: This is 2x leveraged)

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="SDP",currency=="USD"))

Financials Sector: ProShares UltraShort Financials - SKF (Note: This is 2x leveraged)

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="SKF",currency=="USD"))

Industrials Sector: ProShares UltraShort Industrials - SIJ (Note: This is 2x leveraged)

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="SIJ",currency=="USD"))

Materials Sector: ProShares UltraShort Basic Materials - SMN (Note: This is 2x leveraged)

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="SMN",currency=="USD"))

Energy Sector: ProShares UltraShort Oil & Gas - DUG (Note: This is 2x leveraged, and specifically targets the oil and gas industry)

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="DUG",currency=="USD"))

Real Estate Sector: ProShares UltraShort Real Estate - SRS (Note: This is 2x leveraged)

In [ ]:
draw_plots(data = df_adjusted %>% filter(ticker=="SRS",currency=="USD"))